load myhis

In [15]:
import pandas as pd
mystk_his= pd.read_csv('myhistory_acc.csv',index_col=0)
mystk = mystk_his[mystk_his['stkid']=='00878X']
mystk

,date,Qty,stkid,stkdiv,Interest
index,,,,,
10.0,113/02,44.0,00878X,0,0
11.0,113/03,44.0,00878X,0,0
12.0,113/04,45.0,00878X,0,0
13.0,113/05,46.1,00878X,0,0
14.0,113/06,46.1,00878X,0,0


get from twse div data

In [82]:
import numpy as np
def get_twse_div_data():
    datestr = datetime.datetime.now().strftime('%Y%m%d')
    print("https://www.twse.com.tw/exchangeReport/TWT49U?response=csv&strDate=20230101&endDate="+datestr)
    res = requests.get("https://www.twse.com.tw/exchangeReport/TWT49U?response=csv&strDate=20200101&endDate="+datestr)
    df = pd.read_csv(io.StringIO(res.text.replace("=", "")), header=1)
    df = df.dropna(thresh=5).dropna(how='all', axis=1)
    df = df[~df['資料日期'].isnull()]
    years = df['資料日期'].str.split('年').str[0].astype(int) #+ 1911

        #years.loc[df['資料日期'].str[3] != '年'] = np.nan
    years.loc[df['資料日期'].str.find('年') == -1] = np.nan
    years.loc[years > datetime.datetime.now().year] = np.nan
    years.ffill(inplace=True)
    dates = years.astype(int).astype(str) +'/'+ df['資料日期'].str.split('年').str[1].str.replace('月', '/').str.replace('日', '')
    df['date'] = dates #pd.to_datetime(dates, errors='coerce')

    float_name_list = ['除權息前收盤價', '除權息參考價', '權值+息值', '漲停價格',
                        '跌停價格', '開盤競價基準', '減除股利參考價' , '最近一次申報每股 (單位)淨值',
                        '最近一次申報每股 (單位)盈餘']

    df[float_name_list] = df[float_name_list].astype(str).apply(lambda s:s.str.replace(',', '')).astype(float)
    df['twse_divide_ratio'] = df['除權息前收盤價'] / df['開盤競價基準']
    df.to_csv('twse_div_data.csv',index=0)
    return df
twse_div_data = get_twse_div_data()

https://www.twse.com.tw/exchangeReport/TWT49U?response=csv&strDate=20230101&endDate=20240606


In [ ]:
twse_div_data = pd.read_csv('twse_div_data.csv')
twse_div_data
stkinfo = twse_div_data[twse_div_data['股票代號'] == '00878']
stkinfo

In [11]:
from datetime import datetime
import pandas as pd

mystk_his= pd.read_csv('myhistory_test.csv',index_col=0)
mystk_his = mystk_his.reset_index(drop=True)
def update_interest_data_twse(stkid):
    global mystk_his
    myhis_stkid = stkid
    stkid = stkid.replace('X','')
    stkinfo = twse_div_data[twse_div_data['股票代號']==stkid]
    #print(stkinfo)
    mystk_his1 =  mystk_his[mystk_his['stkid']==myhis_stkid]
    for idx in stkinfo.index:
        #print(idx)
        s = stkinfo['date'][idx]
        sdiv = stkinfo['權值+息值'][idx]
        print(sdiv)
        year_month = s.split('/')[0] + '/' + s.split('/')[1]
        print(year_month)
        index = mystk_his1.index[(mystk_his1["date"] == year_month)]
        myminimusDate = mystk_his1["date"].min()
        index = mystk_his1.index[(mystk_his1["date"] == year_month)]
        Myyear = str(int(myminimusDate.split('/')[0])+1911)
        stkyear = str(int(year_month.split('/')[0])+1911)
        stkyear = stkyear + '/' + year_month.split('/')[1]
        Myyear = Myyear + '/' + myminimusDate.split('/')[1]
        # 將字串轉換成日期格式
        date_format = '%Y/%m'
        stkdate1 = datetime.strptime(stkyear, date_format)
        mydate2 = datetime.strptime(Myyear, date_format)
        # 比較兩個日期
        if stkdate1 < mydate2:
            continue
        # mystk日期不存在 就新增一筆
        if len(index) == 0: 
            cumulative_sum = mystk_his1['Qty'].sum()
            new_data = [{'date': year_month, 'Qty': 0, 'stkid': stkid, 'stkdiv': sdiv, 'Interest': 0}]
            new_data[0]['Interest'] =  sdiv *  cumulative_sum *1000
            new_data[0]['stkdiv'] = sdiv
            new_df = pd.DataFrame(new_data)
            mystk_his = pd.concat([mystk_his, new_df], ignore_index=True)
            #mystk = pd.merge(mystk_his,mystk_his1)
            continue

        cumulative_sum = mystk_his1.loc[:index[0], 'Qty'].sum()        
        print(index[0])
        #print(mystk_his1.iloc[index[0]].to_frame().T)
        mystk_his.loc[index[0],'stkdiv'] = sdiv
        mystk_his.loc[index[0],'Interest'] = sdiv *  cumulative_sum * 1000
    mystk_his.to_csv('aaa.csv')
    return mystk_his
update_interest_data_twse('2105')


1.0
109/07
1.2
110/08
1.2
111/07
1.4
112/06
2.0
113/06


,date,Qty,stkid,stkdiv,Interest
0,113/03,150.000,00937B,0.0,0.0
1,113/02,50.000,00751B,0.0,0.0
2,113/03,50.000,00751B,0.0,0.0
3,113/04,53.000,00751B,0.0,0.0
4,113/05,54.519,00751B,0.0,0.0
...,...,...,...,...,...
102,113/09,9.900,5871,0.0,0.0
103,113/10,9.900,5871,0.0,0.0
104,113/11,9.900,5871,0.0,0.0
105,113/12,9.900,5871,0.0,0.0


start from my stkinfo get from otc data

In [26]:
import requests
import pandas as pd
import datetime
import json

datestr = '113/06/04'
def get_otc_div_data():
    y = datetime.datetime.now().year
    m = datetime.datetime.now().month
    d = datetime.datetime.now().day

    y = str(y-1911)
    m = str(m) if m > 9 else '0' + str(m)
    d = str(d) if d > 9 else '0' + str(d)

    datestr = '%s/%s/%s' % (y,m,d)
    res_otc = requests.get('https://www.tpex.org.tw/web/stock/exright/dailyquo/exDailyQ_result.php?l=zh-tw&d=113/01/02&ed=' + datestr)

    df = pd.DataFrame(json.loads(res_otc.text)['aaData'])
    df.columns = ['除權息日期', '代號', '名稱', '除權息前收盤價', '除權息參考價',
                        '權值', '息值',"權+息值","權/息","漲停價格","跌停價格","開盤競價基準",
                        "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                        "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"]


    float_name_list = [ '除權息前收盤價', '除權息參考價',
                            '權值', '息值',"權+息值","漲停價格","跌停價格","開盤競價基準",
                            "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                            "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"
    ]
    df[float_name_list] = df[float_name_list].astype(str).apply(lambda s:s.str.replace(',', '')).astype(float)

    # set stock id
    df['stock_id'] = df['代號'] + ' ' + df['名稱']

    # set dates
    dates = df['除權息日期'].str.split('/')
    dates = (dates.str[0].astype(int) + 1911).astype(str) + '/' + dates.str[1] + '/' + dates.str[2]
    df['date'] = pd.to_datetime(dates)

    df['otc_divide_ratio'] = df['除權息前收盤價'] / df['開盤競價基準']
    df.to_csv('otc_div_data.csv')
    return df
#return df.set_index(['stock_id', 'date'])
get_otc_div_data()

KeyError: 'aaData'

In [ ]:
y = datetime.datetime.now().year
m = datetime.datetime.now().month
d = datetime.datetime.now().day

y = str(y-1911)
m = str(m) if m > 9 else '0' + str(m)
d = str(d) if d > 9 else '0' + str(d)

datestr = '%s/%s/%s' % (y,m,d)
res_otc = requests.get('https://www.tpex.org.tw/web/stock/exright/dailyquo/exDailyQ_result.php?l=zh-tw&d=113/01/02&ed=' + datestr)


df = pd.DataFrame(json.loads(res_otc.text)['tables'][0]['data'])

df.columns = ['除權息日期', '代號', '名稱', '除權息前收盤價', '除權息參考價',
                    '權值', '息值',"權+息值","權/息","漲停價格","跌停價格","開盤競價基準",
                    "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                    "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"]


float_name_list = [ '除權息前收盤價', '除權息參考價',
                        '權值', '息值',"權+息值","漲停價格","跌停價格","開盤競價基準",
                        "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                        "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"
]
df[float_name_list] = df[float_name_list].astype(str).apply(lambda s:s.str.replace(',', '')).astype(float)

# set stock id
df['stock_id'] = df['代號'] + ' ' + df['名稱']

# set dates
dates = df['除權息日期'].str.split('/')
dates = (dates.str[0].astype(int) + 1911).astype(str) + '/' + dates.str[1] + '/' + dates.str[2]
df['date'] = pd.to_datetime(dates)

df['otc_divide_ratio'] = df['除權息前收盤價'] / df['開盤競價基準']
df.to_csv('otc_div_data.csv')
return df

KeyError: 'aaData'

In [23]:
ss = json.loads(res_otc.text)
df = pd.DataFrame(json.loads(res_otc.text)['tables'][0]['data'])

df.columns = ['除權息日期', '代號', '名稱', '除權息前收盤價', '除權息參考價',
                    '權值', '息值',"權+息值","權/息","漲停價格","跌停價格","開盤競價基準",
                    "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                    "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"]

df



,除權息日期,代號,名稱,除權息前收盤價,除權息參考價,權值,息值,權+息值,權/息,漲停價格,...,開盤競價基準,減除股利參考價,現金股利,每千股無償配股,現金增資股數,現金增資認購價,公開承銷股數,員工認購股數,原股東認購數,按持股比例千股認購
0,113/01/03,6629,泰金-KY,55.00,53.50,0.000000,1.500000,1.500000,除息,58.80,...,53.50,53.50,1.50000000,0.00000000,0,0.00,0,0,0,0.00000000
1,113/01/08,6613,朋億*,140.50,137.07,0.000000,3.429363,3.429363,除息,150.50,...,137.00,137.07,3.42936322,0.00000000,0,0.00,0,0,0,0.00000000
2,113/01/10,1799,易威,39.25,38.72,0.528412,0.000000,0.528412,除權,43.15,...,39.25,39.25,0.00000000,0.00000000,6000000,28.60,600000,600000,4800000,41.76517920
3,113/01/11,6763,綠界科技,459.50,450.40,0.000000,9.100000,9.100000,除息,495.00,...,450.50,450.40,9.10000003,0.00000000,0,0.00,0,0,0,0.00000000
4,113/01/15,5536,聖暉*,179.00,174.50,0.000000,4.500000,4.500000,除息,191.50,...,174.50,174.50,4.50000000,0.00000000,0,0.00,0,0,0,0.00000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
964,113/10/29,2061,風青,20.95,20.54,0.409564,0.000000,0.409564,除權,23.00,...,20.95,20.95,0.00000000,0.00000000,10000000,18.60,1000000,1500000,7500000,158.30114754
965,113/10/30,6596,寬宏藝術,73.80,73.30,0.500000,0.000000,0.500000,除權,81.10,...,73.80,73.80,0.00000000,0.00000000,5000000,70.00,500000,750000,3750000,113.63636364
966,113/10/31,00950B,凱基A級公司債,15.06,14.98,0.000000,0.082000,0.082000,除息,9999.95,...,14.98,14.98,0.08200000,0.00000000,0,0.00,0,0,0,0.00000000
967,113/10/31,00959B,大華投等美債15Y+,9.90,9.85,0.000000,0.055000,0.055000,除息,9999.95,...,9.85,9.85,0.05500000,0.00000000,0,0.00,0,0,0,0.00000000


In [25]:
ss = json.loads(res_otc.text)
df = pd.DataFrame(json.loads(res_otc.text)['tables'][0]['data'])

df.columns = ['除權息日期', '代號', '名稱', '除權息前收盤價', '除權息參考價',
                    '權值', '息值',"權+息值","權/息","漲停價格","跌停價格","開盤競價基準",
                    "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                    "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"]

df

,除權息日期,代號,名稱,除權息前收盤價,除權息參考價,權值,息值,權+息值,權/息,漲停價格,...,開盤競價基準,減除股利參考價,現金股利,每千股無償配股,現金增資股數,現金增資認購價,公開承銷股數,員工認購股數,原股東認購數,按持股比例千股認購
0,113/01/03,6629,泰金-KY,55.00,53.50,0.000000,1.500000,1.500000,除息,58.80,...,53.50,53.50,1.50000000,0.00000000,0,0.00,0,0,0,0.00000000
1,113/01/08,6613,朋億*,140.50,137.07,0.000000,3.429363,3.429363,除息,150.50,...,137.00,137.07,3.42936322,0.00000000,0,0.00,0,0,0,0.00000000
2,113/01/10,1799,易威,39.25,38.72,0.528412,0.000000,0.528412,除權,43.15,...,39.25,39.25,0.00000000,0.00000000,6000000,28.60,600000,600000,4800000,41.76517920
3,113/01/11,6763,綠界科技,459.50,450.40,0.000000,9.100000,9.100000,除息,495.00,...,450.50,450.40,9.10000003,0.00000000,0,0.00,0,0,0,0.00000000
4,113/01/15,5536,聖暉*,179.00,174.50,0.000000,4.500000,4.500000,除息,191.50,...,174.50,174.50,4.50000000,0.00000000,0,0.00,0,0,0,0.00000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
964,113/10/29,2061,風青,20.95,20.54,0.409564,0.000000,0.409564,除權,23.00,...,20.95,20.95,0.00000000,0.00000000,10000000,18.60,1000000,1500000,7500000,158.30114754
965,113/10/30,6596,寬宏藝術,73.80,73.30,0.500000,0.000000,0.500000,除權,81.10,...,73.80,73.80,0.00000000,0.00000000,5000000,70.00,500000,750000,3750000,113.63636364
966,113/10/31,00950B,凱基A級公司債,15.06,14.98,0.000000,0.082000,0.082000,除息,9999.95,...,14.98,14.98,0.08200000,0.00000000,0,0.00,0,0,0,0.00000000
967,113/10/31,00959B,大華投等美債15Y+,9.90,9.85,0.000000,0.055000,0.055000,除息,9999.95,...,9.85,9.85,0.05500000,0.00000000,0,0.00,0,0,0,0.00000000


In [ ]:
data = json_data['tables'][0]['data']

{'date': '20240102~20241104',
 'tables': [{'title': '',
   'totalCount': 969,
   'fields': ['除權息日期',
    '代號',
    '名稱',
    '除權息前收盤價',
    '除權息參考價',
    '權值',
    '息值',
    '權值+息值',
    '權/息',
    '漲停價',
    '跌停價',
    '開始交易基準價',
    '減除股利參考價',
    '現金股利',
    '每仟股無償配股',
    '現金增資股數',
    '現金增資認購價',
    '公開承銷股數',
    '員工認購股數',
    '原股東認購股數',
    '按持股比例仟股認購'],
   'data': [['113/01/03',
     '6629',
     '泰金-KY           ',
     '55.00',
     '53.50',
     '0.000000',
     '1.500000',
     '1.500000',
     '除息',
     '58.80',
     '48.15',
     '53.50',
     '53.50',
     '1.50000000',
     '0.00000000',
     '0',
     '0.00',
     '0',
     '0',
     '0',
     '0.00000000'],
    ['113/01/08',
     '6613',
     '朋億*             ',
     '140.50',
     '137.07',
     '0.000000',
     '3.429363',
     '3.429363',
     '除息',
     '150.50',
     '123.50',
     '137.00',
     '137.07',
     '3.42936322',
     '0.00000000',
     '0',
     '0.00',
     '0',
     '0',
     '0',
     '0.00000000'

In [11]:
df.columns = ['除權息日期', '代號', '名稱', '除權息前收盤價', '除權息參考價',
                    '權值', '息值',"權+息值","權/息","漲停價格","跌停價格","開盤競價基準",
                    "減除股利參考價","現金股利", "每千股無償配股", "現金增資股數", "現金增資認購價",
                    "公開承銷股數", "員工認購股數","原股東認購數", "按持股比例千股認購"]

ValueError: Length mismatch: Expected axis has 6 elements, new values have 21 elements

In [16]:
import pandas as pd
otc_div_data = pd.read_csv('otc_div_data.csv')

In [7]:

data = [{'date': '113/02', 'Qty': 0, 'stkid': '00937B','stkdiv':0,'Interest':0},
        {'date': '113/03', 'Qty': 165, 'stkid': '00937B','stkdiv':0,'Interest':0},
        {'date': '113/04', 'Qty': 170, 'stkid': '00937B','stkdiv':0,'Interest':0},
        {'date': '113/05', 'Qty': 170, 'stkid': '00937B','stkdiv':0,'Interest':0},
        {'date': '113/06', 'Qty': 232, 'stkid': '00937B','stkdiv':0,'Interest':0},
        {'date': '113/07', 'Qty': 232, 'stkid': '00937B','stkdiv':0,'Interest':0},
        {'date': '113/08', 'Qty': 232, 'stkid': '00937B','stkdiv':0,'Interest':0}]

mystk_his = pd.DataFrame(data)
print(mystk_his)
mystk_his.to_csv('myhistory.csv')

     date  Qty   stkid  stkdiv  Interest
0  113/02    0  00937B       0         0
1  113/03  165  00937B       0         0
2  113/04  170  00937B       0         0
3  113/05  170  00937B       0         0
4  113/06  232  00937B       0         0
5  113/07  232  00937B       0         0
6  113/08  232  00937B       0         0


In [51]:
from datetime import datetime
def update_interest_data(stkid):
    mystk_his= pd.read_csv('myhistory_test.csv',index_col=0)
    mystk_his = mystk_his.reset_index(drop=True)
    myhis_stkid = stkid
    stkid = stkid.replace('X','')
    stkinfo = otc_div_data[otc_div_data['代號']==myhis_stkid]
    if len(stkinfo) == 0: #not found in otc data
        return update_interest_data_twse(myhis_stkid)
    #print(stkinfo)
    mystk_his1 =  mystk_his[mystk_his['stkid']==stkid]
    cumulative_sum = 0
    for idx in stkinfo.index:
        #print(idx)
        s = stkinfo['除權息日期'][idx]
        sdiv = stkinfo['息值'][idx]
        print(sdiv)
        year_month = s.split('/')[0] + '/' + s.split('/')[1]
        print(year_month)
        #做日期處理 比大小 ,以前的日期 不處理
        myminimusDate = mystk_his1["date"].min()
        index = mystk_his1.index[(mystk_his1["date"] == year_month)]
        Myyear = str(int(myminimusDate.split('/')[0])+1911)
        stkyear = str(int(year_month.split('/')[0])+1911)
        stkyear = stkyear + '/' + year_month.split('/')[1]
        Myyear = Myyear + '/' + myminimusDate.split('/')[1]
        # 將字串轉換成日期格式
        date_format = '%Y/%m'
        stkdate1 = datetime.strptime(stkyear, date_format)
        mydate2 = datetime.strptime(Myyear, date_format)

        # 比較兩個日期
        if stkdate1 < mydate2:
            continue
        # mystk日期不存在 就新增一筆
        if len(index) == 0: 
            new_data = [{'date': year_month, 'Qty': 0, 'stkid': stkid, 'stkdiv': sdiv, 'Interest': 0}]
            new_data[0]['Interest'] =  sdiv *  cumulative_sum *1000
            new_data[0]['stkdiv'] = sdiv
            new_df = pd.DataFrame(new_data)
            mystk_his = pd.concat([mystk_his, new_df], ignore_index=True)
            #mystk = pd.merge(mystk_his,mystk_his1)
            continue
            #mystk_his.loc[mystk_his['stkid'] == myhis_stkid, :] = mystk_his1
            #mystk_his1 =  mystk_his[mystk_his['stkid']==myhis_stkid]
            #index = mystk_his1.index[(mystk_his1["date"] == year_month)]
            #continue
        cumulative_sum = mystk_his1.loc[:index[0], 'Qty'].sum()   
        print(index[0])
        #print(mystk_his1.iloc[index[0]].to_frame().T)
        mystk_his.loc[index[0],'stkdiv'] = sdiv
        mystk_his.loc[index[0],'Interest'] = sdiv *  cumulative_sum *1000

    #mystk_his.loc[mystk_his['stkid'] == myhis_stkid, :] = mystk_his1
    #mystk_his['stkdiv'] = mystk_his1['stkdiv']
    #mystk_his['Interest'] = mystk_his1['Interest']
    #mystk_his.to_csv('aaa.csv')
    
    return mystk_his

update_interest_data('00937B')    

0.084
113/02
0.084
113/03
2.0
0.084
113/04
0.084
113/05


,date,Qty,stkid,stkdiv,Interest
0,113/03,150.000,00937B,0.084,12600.0
1,113/02,50.000,00751B,0.000,0.0
2,113/03,50.000,00751B,0.000,0.0
3,113/04,53.000,00751B,0.000,0.0
4,113/05,54.519,00751B,0.000,0.0
...,...,...,...,...,...
123,113/10,9.900,5871,0.000,0.0
124,113/11,9.900,5871,0.000,0.0
125,113/12,9.900,5871,0.000,0.0
126,113/04,0.000,00937B,0.084,12600.0


In [ ]:
new_data = [{'date': 'year_month', 'Qty': 0, 'stkid': 'stkid', 'stkdiv': 'sdiv', 'Interest': 0}]
new_data[0]['stkdiv'] = 100
new_data

In [ ]:
mystk_his= pd.read_csv('myhistory.csv',index_col=0)
mystk_his = mystk_his.reset_index(drop=True)
ids = mystk_his.value_counts('stkid')
for a in ids.index:
    print(a)
    mystk_his = update_interest_data(a)
mystk_his.to_csv('result.csv')
mystk_his.head(20)

In [ ]:
stkinfo = twse_div_data[twse_div_data['股票代號']=='5871']
stkinfo

In [24]:
import pandas as pd
hisdf = pd.read_csv('myhistory.csv')
del hisdf['Unnamed: 0']
hisdf = hisdf.reset_index(drop=True)

hisdf
hisdf.groupby(['stkid']).agg({'Qty': 'sum','Interest':'sum'})


,Qty,Interest
stkid,,
0056X,13.630,10507.00000
00687B,5.000,1700.00000
00713X,8.571,19016.50000
00751B,54.519,47623.93000
00857B,4.000,840.00000
00878X,46.815,40822.85000
00882X,30.942,0.00000
00933B,36.000,3362.00000
00937B,235.000,66696.00000


In [33]:

writer = pd.ExcelWriter('output.xlsx', engine='xlsxwriter')

s = hisdf.groupby(['date','stkid']).agg({'Qty': 'sum','Interest':'sum'})
#s.to_csv('sumbydate.csv')
s.to_excel(writer, sheet_name='sumbydate_stkid')

s = hisdf.groupby(['stkid']).agg({'Qty': 'sum','Interest':'sum'})
s.to_excel(writer, sheet_name='sumbystkid')

s = hisdf.groupby(['date']).agg({'Interest':'sum'})
s.to_excel(writer, sheet_name='sumbydate')
writer.close()

In [ ]:
s = hisdf.groupby(['date','stkid']).sum(['Interest','Qty'])
s.to_csv('sumbydate.csv')

s = hisdf.groupby(['stkid']).sum(['Qty'])
hisdf
#s.to_csv('sumbyst.csv')

In [2]:
import requests

def get_stock_price(stock_symbol):
    url = f"https://www.twse.com.tw/exchangeReport/STOCK_DAY_AVG?response=json&date=20210910&stockNo={stock_symbol}"
    response = requests.get(url)
    data = response.json()
    
    if data['stat'] == 'OK':
        price = data['data'][0][1]  # 获取当日收盘价
        volume = data['data'][0][2]  # 获取当日成交量
        print(f"Stock Price for {stock_symbol}: {price}")
        print(f"Volume for {stock_symbol}: {volume}")
    else:
        print("Failed to fetch data from TWSE")

if __name__ == "__main__":
    stock_symbol = "2330"  # 台積電代號
    get_stock_price(stock_symbol)


KeyboardInterrupt: 